# Kokoro Voice Studio — Kaggle + Cloudflare

A password-protected text-to-speech studio that downloads Kokoro model files from Hugging Face and runs inference inside this Kaggle notebook. Cloudflare Quick Tunnel exposes the browser studio and agent API while the notebook is running.

**Before running:** enable Notebook Internet. A GPU accelerator is optional; this ONNX setup can run on CPU. Run the cells from the top so the inference package and Hugging Face model files are installed/downloaded before launching the app. Keep the session password private.

In [ ]:
%pip -q install -U "kokoro-onnx>=0.6.1" huggingface_hub gradio soundfile fastapi uvicorn
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /kaggle/working/cloudflared
!chmod +x /kaggle/working/cloudflared
print('Kokoro ONNX runtime and Cloudflare Tunnel are ready.')

In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download
from kokoro_onnx import Kokoro

MODEL_REPO = 'ExNihilus/kokoro-onnx'
MODEL_DIR = Path('/kaggle/working/kokoro-model')
MODEL_DIR.mkdir(parents=True, exist_ok=True)
print(f'Downloading Kokoro model files from Hugging Face: {MODEL_REPO}')
model_path = hf_hub_download(repo_id=MODEL_REPO, filename='kokoro-v1.0.onnx', local_dir=MODEL_DIR)
voices_path = hf_hub_download(repo_id=MODEL_REPO, filename='voices-v1.0.bin', local_dir=MODEL_DIR)
tts = Kokoro(model_path, voices_path)
print('Kokoro model is loaded in this Kaggle runtime.')


In [ ]:
import base64
import secrets
import socket
import subprocess
import tempfile
import threading
import uuid
from pathlib import Path

import gradio as gr
import soundfile as sf
import uvicorn
from fastapi import FastAPI, Header, HTTPException
from fastapi.responses import FileResponse
from pydantic import BaseModel, Field
from starlette.background import BackgroundTask

MAX_CHARACTERS = 1800
SAMPLE_RATE = 24000
VOICE_CHOICES = {
    'American English': ['af_heart','af_bella','af_nicole','af_sarah','af_sky','af_aoede','af_kore','af_alloy','af_jessica','af_nova','af_river','am_adam','am_michael','am_fenrir','am_puck','am_echo','am_eric','am_liam','am_onyx','am_santa'],
    'British English': ['bf_emma','bf_isabella','bf_alice','bf_lily','bm_george','bm_lewis','bm_daniel','bm_fable'],
}
VOICE_LANGUAGE = {voice: language for language, voices in VOICE_CHOICES.items() for voice in voices}
LANGUAGE_CODE = {'American English': 'en-us', 'British English': 'en-gb'}
generation_lock = threading.Lock()
with socket.socket() as port_probe:
    port_probe.bind(('127.0.0.1', 0))
    PORT = port_probe.getsockname()[1]

def generate_speech(text, speaker, language, speed):
    text = (text or '').strip()
    if not text:
        raise gr.Error('Write a line of text before generating speech.')
    if len(text) > MAX_CHARACTERS:
        raise gr.Error(f'Keep each generation under {MAX_CHARACTERS:,} characters.')
    if speaker not in VOICE_LANGUAGE:
        raise gr.Error('Choose a supported Kokoro voice.')
    if VOICE_LANGUAGE[speaker] != language:
        raise gr.Error(f'{speaker} is a {VOICE_LANGUAGE[speaker]} voice. Choose a voice for {language}.')
    try:
        with generation_lock:
            audio, sample_rate = tts.create(
                text, voice=speaker, speed=float(speed), lang=LANGUAGE_CODE[language]
            )
    except Exception as exc:
        raise gr.Error(f'Kokoro generation failed: {exc}') from exc
    output = Path(tempfile.gettempdir()) / f'kokoro-voice-{uuid.uuid4().hex}.wav'
    sf.write(output, audio, sample_rate, subtype='PCM_16')
    return str(output), str(output)

css = '''
#studio {max-width: 1120px; margin: 0 auto;}
.hero {padding: 24px 4px 12px;}
.hero h1 {font-size: 2.55rem; letter-spacing: -0.05em; margin: 0 0 6px;}
.hero p {font-size: 1rem; opacity: .76;}
.generate button {min-height: 52px; font-size: 1rem; font-weight: 700;}
'''
theme = gr.themes.Soft(primary_hue='violet', secondary_hue='slate')
def voices_for_language(language):
    return gr.Dropdown(choices=VOICE_CHOICES[language], value=VOICE_CHOICES[language][0])

with gr.Blocks(title='Kokoro Voice Studio') as demo:
    with gr.Column(elem_id='studio'):
        gr.HTML('<div class="hero"><h1>Kokoro Voice Studio</h1><p>Write a line. Choose a voice. Hear it come alive.</p></div>')
        with gr.Row(equal_height=True):
            with gr.Column(scale=6):
                script = gr.Textbox(label='Your script', placeholder='Paste or write something to say …', lines=9, max_lines=18, max_length=MAX_CHARACTERS, value='Kokoro is an open-weight text-to-speech model running locally in this Kaggle notebook.')
                count = gr.Markdown('**0** characters')
                script.input(lambda value: f'**{len(value or [])}** characters', inputs=script, outputs=count)
                with gr.Row():
                    language = gr.Dropdown(choices=list(VOICE_CHOICES), value='American English', label='Language')
                    voice = gr.Dropdown(choices=VOICE_CHOICES['American English'], value='af_heart', label='Voice')
                language.change(voices_for_language, inputs=language, outputs=voice)
                speed = gr.Slider(minimum=0.7, maximum=1.3, value=1.0, step=0.05, label='Speaking speed')
                gr.Markdown('For pronunciation, add inline IPA hints such as `[Kokoro](/kˈOkəɹO/)`.')
                create = gr.Button('Generate speech', variant='primary', elem_classes=['generate'])
            with gr.Column(scale=5):
                gr.Markdown('### Your audio\nKokoro runs in this Kaggle session. The first generation may take longer while the model warms up.')
                preview = gr.Audio(label='Preview', type='filepath', interactive=False)
                download = gr.File(label='Download WAV', interactive=False)
                gr.Markdown('Choose a preset voice and speaking speed to shape delivery.')
        create.click(generate_speech, inputs=[script, voice, language, speed], outputs=[preview, download])

ACCESS_PASSWORD = secrets.token_urlsafe(12)
api_app = FastAPI(title='Kokoro Voice API', docs_url='/api/docs', redoc_url=None)
class SpeechRequest(BaseModel):
    text: str = Field(min_length=1, max_length=MAX_CHARACTERS)
    speaker: str = 'af_heart'
    language: str = 'American English'
    speed: float = Field(default=1.0, ge=0.7, le=1.3)
    direction: str | None = Field(default=None, description='Deprecated compatibility field; use preset voices, IPA in text, and speed.')

def require_api_password(authorization: str | None = Header(default=None)):
    valid = False
    if authorization and authorization.startswith('Bearer '):
        valid = secrets.compare_digest(authorization[7:], ACCESS_PASSWORD)
    elif authorization and authorization.startswith('Basic '):
        try:
            username, password = base64.b64decode(authorization[6:]).decode('utf-8').split(':', 1)
            valid = username == 'voice' and secrets.compare_digest(password, ACCESS_PASSWORD)
        except (ValueError, UnicodeDecodeError):
            valid = False
    if not valid:
        raise HTTPException(status_code=401, detail='Use the studio password as a Bearer token or HTTP Basic password.', headers={'WWW-Authenticate': 'Basic'})

@api_app.post('/api/v1/tts', response_class=FileResponse, response_description='Generated WAV audio')
def create_speech(payload: SpeechRequest, authorization: str | None = Header(default=None)):
    require_api_password(authorization)
    if payload.speaker not in VOICE_LANGUAGE:
        raise HTTPException(status_code=400, detail='Unknown Kokoro speaker. See /api/docs for supported voice IDs.')
    if VOICE_LANGUAGE[payload.speaker] != payload.language:
        raise HTTPException(status_code=400, detail=f"Speaker {payload.speaker} is for {VOICE_LANGUAGE[payload.speaker]}, not {payload.language}.")
    try:
        audio_path, _ = generate_speech(payload.text, payload.speaker, payload.language, payload.speed)
    except gr.Error as exc:
        raise HTTPException(status_code=400, detail=str(exc)) from exc
    return FileResponse(audio_path, media_type='audio/wav', filename='kokoro-speech.wav', background=BackgroundTask(Path(audio_path).unlink, missing_ok=True))

app = gr.mount_gradio_app(api_app, demo, path='/', auth=('voice', ACCESS_PASSWORD), show_error=True, theme=theme, css=css)
server = uvicorn.Server(uvicorn.Config(app, host='127.0.0.1', port=PORT, log_level='warning'))
threading.Thread(target=server.run, daemon=True).start()
tunnel = subprocess.Popen(
    ['/kaggle/working/cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{PORT}'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1
)
def show_tunnel_logs():
    for line in tunnel.stdout:
        print(line.rstrip(), flush=True)
threading.Thread(target=show_tunnel_logs, daemon=True).start()
print('Studio login username: voice')
print('Studio access password (keep private):', ACCESS_PASSWORD)
print('The temporary https://trycloudflare.com URL will appear in the tunnel logs above.')
print('API: POST https://<your-tunnel>/api/v1/tts (JSON in, WAV audio out)')
print('API docs: https://<your-tunnel>/api/docs')
print('Model files downloaded from Hugging Face:', MODEL_REPO)
print('Keep this cell running. Stop the Kaggle session to close the public URL.')


## Access and safety

Cloudflare Quick Tunnels are temporary demo URLs. The studio requires username `voice` and a random session password; the API accepts that password as a Bearer token or HTTP Basic password. Keep the URL and password private. Speech generation runs locally in this Kaggle notebook after model files are downloaded from Hugging Face. Stop the notebook session to end access.

Model files: [Kokoro ONNX model and voice pack on Hugging Face](https://huggingface.co/ExNihilus/kokoro-onnx) · [Kokoro voices](https://huggingface.co/hexgrad/Kokoro-82M/blob/main/VOICES.md) · [Kokoro ONNX runtime](https://github.com/thewh1teagle/kokoro-onnx) · [Cloudflare Quick Tunnels](https://developers.cloudflare.com/tunnel/get-started/quick-tunnels/)